In [18]:
from bs4 import BeautifulSoup, SoupStrainer
import requests
from langchain_community.document_loaders import WebBaseLoader, SitemapLoader
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from urllib.parse import urljoin, urlparse
import pickle
import os

In [2]:
parse_only = SoupStrainer('div', class_='document')

loader = WebBaseLoader(
    web_paths=("https://docs.manim.community/en/stable/",),
    bs_kwargs=dict(parse_only=parse_only)
)

docs = loader.load()

In [3]:
docs

[Document(page_content='', metadata={'source': 'https://docs.manim.community/en/stable/'})]

In [30]:
def get_all_links(url, base_url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    for link in soup.find_all('a', href=True):
        href = link['href']
        full_url = urljoin(url, href)
        if full_url.startswith(base_url) and full_url != url:
            yield full_url

def crawl_site(start_url, base_url):
    visited = set()
    to_visit = [start_url]
    
    while to_visit:
        current_url = to_visit.pop(0)
        if current_url not in visited and current_url.startswith(base_url):
            # print(f"Crawling: {current_url}")
            visited.add(current_url)
            to_visit.extend(link for link in get_all_links(current_url, base_url) if link not in visited)
    
    return list(visited)


In [38]:
parse_only = SoupStrainer('div', class_='document')


In [41]:
def scrape_manim_docs(start_url):
    # base_url = f"{urlparse(start_url).scheme}://{urlparse(start_url).netloc}"
    all_urls = crawl_site(start_url, start_url)
    # Define SoupStrainer to parse only the main content
    # parse_only = SoupStrainer('div', class_='document')

    # def extract_content(html):
    #     soup = BeautifulSoup(html, 'html.parser')
    #     article = soup.find('article')
    #     return article.get_text(strip=True) if article else None
    
    # loader = WebBaseLoader(
    #     web_paths=all_urls,
    #     bs_kwargs={"parse_only": extract_content},
    # )


    # documents = loader.load()

    documents = []
    for url in all_urls: 
        print(f"Processing URL: {url}")
        response = requests.get(url)        
        soup = BeautifulSoup(response.text, 'html.parser')

        article = soup.find('article')
        
        if article:
            content = article.get_text(strip=True)

            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200,
                length_function=len
            )

            chunks = text_splitter.split_text(content)

            for chunk in chunks:
                documents.append(Document(page_content=chunk, metadata={"source": url}))

            print(f"Processed {len(chunks)} chunks from {url}")
        else:
            print(f"No article found in {url}")

    
    
    print(f"Total Documents created: {len(documents)}")
    return documents
    # loader = WebBaseLoader(
    #     web_paths=all_urls,
    #     bs_kwargs={"parse_only": parse_only}
    # )
    # documents = loader.load()
    # return documents

    
    # Split documents
    # text_splitter = RecursiveCharacterTextSplitter(
    #     chunk_size=1000,
    #     chunk_overlap=200,
    #     length_function=len
    # )
    
    # split_docs = text_splitter.split_documents(documents)
    # return split_docs

In [42]:
docs = scrape_manim_docs("https://docs.manim.community/en/stable/tutorials/")

Processing URL: https://docs.manim.community/en/stable/tutorials/building_blocks.html#furo-main-content
Processed 38 chunks from https://docs.manim.community/en/stable/tutorials/building_blocks.html#furo-main-content
Processing URL: https://docs.manim.community/en/stable/tutorials/building_blocks.html#animateexample
Processed 38 chunks from https://docs.manim.community/en/stable/tutorials/building_blocks.html#animateexample
Processing URL: https://docs.manim.community/en/stable/tutorials/building_blocks.html#transforming-mobjects-into-other-mobjects
Processed 38 chunks from https://docs.manim.community/en/stable/tutorials/building_blocks.html#transforming-mobjects-into-other-mobjects
Processing URL: https://docs.manim.community/en/stable/tutorials/quickstart.html#positioning-mobjects
Processed 13 chunks from https://docs.manim.community/en/stable/tutorials/quickstart.html#positioning-mobjects
Processing URL: https://docs.manim.community/en/stable/tutorials/index.html#furo-main-content


In [44]:
for i, doc in enumerate(docs[:5]): # Debugging
    print(f"Document {i + 1}:")
    print(f"Content length: {len(doc.page_content)}")
    print(f"Content preview: {doc.page_content[:500]}")  
    print(f"Metadata: {doc.metadata}")
    print("---")

Document 1:
Content length: 890
Content preview: Manim’s building blocks¶This document explains the building blocks of manim and will give you all the
necessary tools to start producing your own videos.Essentially, manim puts at your disposal three different concepts that you can
orchestrate together to produce mathematical animations: themathematical object(ormobjectfor short), theanimation, and thescene.  As we will see in the following sections, each of these three
concepts is implemented in manim as a separate class: theMobject,Animation, 
Metadata: {'source': 'https://docs.manim.community/en/stable/tutorials/building_blocks.html#furo-main-content'}
---
Document 2:
Content length: 954
Content preview: that derives fromMobjectrepresents an object that can be displayed
on the screen.  For example, simple shapes such asCircle,Arrow, andRectangleare all mobjects.  More complicated
constructs such asAxes,FunctionGraph, orBarChartare mobjects as well.If you try to display an instance ofM

In [45]:
def save_documents(documents, filename):
    with open(filename, 'wb') as f:
        pickle.dump(documents, f)
    print(f"Documents saved to {filename}")


save_documents(docs, "manim_docs.pkl")

Documents saved to manim_docs.pkl


In [46]:
def load_documents(filename):
    with open(filename, 'rb') as f:
        documents = pickle.load(f)
    print(f"Documents loaded from {filename}")
    return documents

loaded_docs = load_documents("manim_docs.pkl")

Documents loaded from manim_docs.pkl


In [48]:
loaded_docs[:3]

[Document(page_content='Manim’s building blocks¶This document explains the building blocks of manim and will give you all the\nnecessary tools to start producing your own videos.Essentially, manim puts at your disposal three different concepts that you can\norchestrate together to produce mathematical animations: themathematical object(ormobjectfor short), theanimation, and thescene.  As we will see in the following sections, each of these three\nconcepts is implemented in manim as a separate class: theMobject,Animation, andSceneclasses.NoteIt is recommended that you read the tutorialsQuickstartandManim’s Output Settingsbefore reading this page.Mobjects¶Mobjects are the basic building blocks for all manim animations.  Each class\nthat derives fromMobjectrepresents an object that can be displayed\non the screen.  For example, simple shapes such asCircle,Arrow, andRectangleare all mobjects.  More complicated', metadata={'source': 'https://docs.manim.community/en/stable/tutorials/building

In [51]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain_google_vertexai import (
    VertexAI,
    ChatVertexAI,
    VertexAIEmbeddings,
    VectorSearchVectorStore
)
model_name = "hkunlp/instructor-xl"

hf_embeddings = HuggingFaceEmbeddings(model_name=model_name)

vectorstore = Chroma.from_documents(documents=loaded_docs, embeddings=hf_embeddings)


/Users/jamessong/Desktop/an-gen/.venv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
/Users/jamessong/Desktop/an-gen/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
